In [0]:
%run "../common functions"

In [0]:
from pyspark.sql.functions import collect_list, col

In [0]:
%sql
insert overwrite raw_catalogue.jobs.linkedJobs select a.* from raw_catalogue.jobs.linkedJobs a left anti join main_catalogue.jobs.curated_jobs b on split_part(a.url,"?",1) = split_part(b.url,"?",1)

In [0]:
dbutils.notebook.run("../filtering jobs", 0, {"process": "linkedJobs"})

In [0]:
df = spark.table('main_catalogue.jobs.users')
chat_ids = df.filter(df.skill == "Data Engineer").select(collect_list('chat_id')).collect()[0][0]

In [0]:
linkedin_jobs = spark.table("staging_catalogue.jobs.linkedJobs") \
                .select('url', 'title', 'company', 'location', 'hrs_ago', 'score') \
                .orderBy(col("score").desc())

In [0]:
form_message_and_send(linkedin_jobs, chat_ids)

In [0]:
%sql
insert into main_catalogue.jobs.curated_jobs
select url, title, company, location, hrs_ago, score from staging_catalogue.jobs.linkedJobs